In [1]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, models
from torch import nn
import os

print("All libraries loaded!")
print(f"GPU: {torch.cuda.get_device_name(0)}")

All libraries loaded!
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import os

# Download dataset from Kaggle
os.system('kaggle datasets download -d omkargurav/face-mask-dataset')
print("Download complete!")

Download complete!


In [4]:
import zipfile
import os

# Extract the dataset
with zipfile.ZipFile('face-mask-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('face_mask_data')

# Check what's inside
for folder in os.listdir('face_mask_data'):
    path = os.path.join('face_mask_data', folder)
    if os.path.isdir(path):
        count = len(os.listdir(path))
        print(f"{folder}: {count} images")

data: 2 images


In [5]:
# Explore full folder structure
for root, dirs, files in os.walk('face_mask_data'):
    level = root.replace('face_mask_data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if files:
        print(f'{indent}  → {len(files)} files')

face_mask_data/
  data/
    without_mask/
      → 3828 files
    with_mask/
      → 3725 files


In [6]:
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch

# 1. Data preparation
data_dir = 'face_mask_data/data'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(data_dir, transform=transform)
classes = dataset.classes
print(f"Classes: {classes}")
print(f"Total images: {len(dataset)}")

# 2. Split into train/test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

print(f"Training: {train_size} | Testing: {test_size}")

# 3. Load pretrained MobileNetV2
device = torch.device('cuda')
model = models.mobilenet_v2(weights='DEFAULT')
model.classifier[1] = nn.Linear(1280, 2)
model = model.to(device)

print(f"\nModel loaded on: {device}")
print("Ready to train!")

Classes: ['with_mask', 'without_mask']
Total images: 7553
Training: 6042 | Testing: 1511


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\rahul/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth
100%|██████████| 13.6M/13.6M [00:03<00:00, 3.87MB/s]



Model loaded on: cuda
Ready to train!


In [7]:
import torch.optim as optim

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train for 5 epochs
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.3f} | Accuracy: {acc:.2f}%")

print("\n✅ Training Complete!")
torch.save(model.state_dict(), 'face_mask_model.pth')
print("Model saved!")

C:\Users\rahul\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/5 | Loss: 0.062 | Accuracy: 97.85%
Epoch 2/5 | Loss: 0.031 | Accuracy: 98.96%
Epoch 3/5 | Loss: 0.021 | Accuracy: 99.27%
Epoch 4/5 | Loss: 0.019 | Accuracy: 99.37%
Epoch 5/5 | Loss: 0.027 | Accuracy: 99.12%

✅ Training Complete!
Model saved!


In [10]:
import cv2
import torch
from PIL import Image
from torchvision import transforms

# Load the saved model
device = torch.device('cuda')
model.eval()

# Transform for each frame
cam_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

# Start webcam
cap = cv2.VideoCapture(0)
print("Webcam started! Press 'Q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert frame to PIL and predict
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = cam_transform(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        _, pred = output.max(1)
        label = classes[pred.item()]
        confidence = torch.softmax(output, dim=1).max().item() * 100
    
    # Draw result on frame
    color = (0, 255, 0) if label == 'with_mask' else (0, 0, 255)
    text = f"{label} ({confidence:.1f}%)"
    cv2.putText(frame, text, (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    cv2.rectangle(frame, (0,0), (frame.shape[1]-1, frame.shape[0]-1), color, 3)
    
    cv2.imshow('Face Mask Detector - Press Q to quit', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Webcam closed!")

Webcam started! Press 'Q' to quit.
Webcam closed!
